# BioPred Phase 1 -- Feature QA, Scaffold Split Strategy, and Modeling Handoff

## Purpose

This notebook converts the frozen molecular feature artifacts from Notebook 06 into split-aware, modeling-ready artifacts for downstream machine learning.

Notebook 06 generated the structure-derived feature matrix, RDKit descriptors, Morgan fingerprints, labels, metadata, and featurization report. Notebook 07 does not redo featurization. Its role is to validate feature readiness, create a leakage-aware train/test split, fit preprocessing on training data only, and save split-aware modeling artifacts.

The central rule is simple: any preprocessing step that learns from feature distributions must be fit on training rows only. Full-dataset preprocessing is not allowed.

## Inputs

- Notebook 06 feature artifacts under:
  - `data/processed/features/`

Expected inputs include:

- combined feature matrix `X`
- RDKit descriptor table
- Morgan fingerprint table
- primary target: `active_median_px_6_0`
- sensitivity targets: `active_median_px_5_5`, `active_median_px_6_5`
- modeling metadata
- featurization report JSON

## Outputs

This notebook will save:

- scaffold assignments
- train/test split assignments
- raw train/test feature, target, sensitivity-label, and metadata tables
- fitted train-only preprocessing artifact
- processed train/test modeling matrices
- feature schema JSON
- split QA report JSON
- modeling handoff policy for Notebook 08

## Objectives

1. Load the frozen feature artifacts created in Notebook 06.
2. Confirm that the loaded artifacts are aligned, complete, and organized into the expected feature blocks.
3. Carry forward the focused feature-quality checks needed for modeling, including missing-value review and row-level QA flags.
4. Create a scaffold-aware training/test split so structurally related molecules are not split across the final evaluation boundary.
5. Validate the locked split by checking molecule counts, label balance, scaffold overlap, scaffold concentration, and feature-QA flag distribution.
6. Define how downstream modeling should handle RDKit descriptors and Morgan fingerprints without fitting preprocessing artifacts in this notebook.
7. Save the raw train/test features, labels, metadata, scaffolds, QA flags, and feature schema needed by Notebook 08.
8. Save a modeling handoff policy that requires model comparison and tuning to happen only inside the training set using scaffold-aware cross-validation.



In [ ]:
# list imports needed for this notebook
from pathlib import Path
import sys
import os
import json
import pandas as pd
import numpy as np
import warnings
import joblib
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

warnings.filterwarnings("ignore")

# Force notebook runtime to project root
%cd /home/ryanm/projects/BioPred

# define paths and link src.paths
SRC_DIR = Path.cwd() / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import biopred.paths as paths

/home/ryanm/projects/BioPred


In [2]:
# check paths so we can upload artifacts (paths were just made in paths.py)
print(f"Feature artifact directory: {paths.FEATURES_DIR}")
print(f"Modeling artifact directory: {paths.MODELING_DIR}")

Feature artifact directory: /home/ryanm/projects/BioPred/data/processed/features
Modeling artifact directory: /home/ryanm/projects/BioPred/data/processed/modeling


In [3]:
# create the artifact_manifest, where we prep the artifacts and check their paths before reading them in
artifact_manifest = {
    "X" : paths.FEATURES_DIR / "chembl_gabaa_feature_matrix.parquet",
    "rdkit_descriptors" : paths.FEATURES_DIR / "chembl_gabaa_rdkit_descriptors.parquet",
    "morgan_fingerprints" : paths.FEATURES_DIR / "chembl_gabaa_morgan_fingerprints.parquet",
    "y_primary" : paths.FEATURES_DIR / "chembl_gabaa_primary_target.parquet",
    "y_sensitivity" : paths.FEATURES_DIR / "chembl_gabaa_sensitivity_labels.parquet",
    "metadata" : paths.FEATURES_DIR / "chembl_gabaa_modeling_metadata.parquet",
    "featurization_report" : paths.FEATURES_DIR / "chembl_gabaa_featurization_report.json"
}

artifact_manifest

{'X': PosixPath('/home/ryanm/projects/BioPred/data/processed/features/chembl_gabaa_feature_matrix.parquet'),
 'rdkit_descriptors': PosixPath('/home/ryanm/projects/BioPred/data/processed/features/chembl_gabaa_rdkit_descriptors.parquet'),
 'morgan_fingerprints': PosixPath('/home/ryanm/projects/BioPred/data/processed/features/chembl_gabaa_morgan_fingerprints.parquet'),
 'y_primary': PosixPath('/home/ryanm/projects/BioPred/data/processed/features/chembl_gabaa_primary_target.parquet'),
 'y_sensitivity': PosixPath('/home/ryanm/projects/BioPred/data/processed/features/chembl_gabaa_sensitivity_labels.parquet'),
 'metadata': PosixPath('/home/ryanm/projects/BioPred/data/processed/features/chembl_gabaa_modeling_metadata.parquet'),
 'featurization_report': PosixPath('/home/ryanm/projects/BioPred/data/processed/features/chembl_gabaa_featurization_report.json')}

In [4]:
# now read in the artifacts so we can employ them in this notebook
X = pd.read_parquet(artifact_manifest["X"])
rdkit_descriptors = pd.read_parquet(artifact_manifest["rdkit_descriptors"])
morgan_fingerprints = pd.read_parquet(artifact_manifest["morgan_fingerprints"])
y_primary = pd.read_parquet(artifact_manifest["y_primary"])
y_sensitivity = pd.read_parquet(artifact_manifest["y_sensitivity"])
metadata = pd.read_parquet(artifact_manifest["metadata"])

with open(artifact_manifest["featurization_report"], "r") as f:
    featurization_report = json.load(f)

print("Notebook 06 artifacts loaded.")


Notebook 06 artifacts loaded.


In [5]:
# quick verification on shape before moving on.
loaded_artifacts = {
    "X": X,
    "rdkit_descriptors": rdkit_descriptors,
    "morgan_fingerprints": morgan_fingerprints,
    "y_primary": y_primary,
    "y_sensitivity": y_sensitivity,
    "metadata": metadata,
}

pd.DataFrame(
    [
        {
            "artifact" : name,
            "shape" : obj.shape,
        }
        for name, obj in loaded_artifacts.items()
    ]
)

,artifact,shape
0,X,"(1546, 2265)"
1,rdkit_descriptors,"(1546, 217)"
2,morgan_fingerprints,"(1546, 2048)"
3,y_primary,"(1546, 1)"
4,y_sensitivity,"(1546, 2)"
5,metadata,"(1546, 12)"


In [6]:
# to check row-alignment we will just do one cell with assertion
# this is just to make sure for any downstream bug prevention
assert X.index.equals(rdkit_descriptors.index), "X and RDKit descriptor rows are misaligned."
assert X.index.equals(morgan_fingerprints.index), "X and Morgan fingerprint rows are misaligned."
assert X.index.equals(y_primary.index), "X and primary target rows are misaligned."
assert X.index.equals(y_sensitivity.index), "X and sensitivity-label rows are misaligned."
assert X.index.equals(metadata.index), "X and metadata rows are misaligned."

print("Row alignment checks passed.")

Row alignment checks passed.


### **Section 3: Focused Feature QA**

This section checks whether the loaded molecular feature artifacts are safe to hand off for scaffold splitting and downstream modeling.

The goal is not broad EDA. The goal is to identify feature-generation failures, forbidden-column leakage, invalid values, constant features, Morgan fingerprint encoding issues, and valid-but-extreme molecular feature values that should be flagged for downstream review.

In [32]:
# creating reusable feature blocks
rdkit_feature_cols = rdkit_descriptors.columns.tolist()
morgan_feature_cols = morgan_fingerprints.columns.tolist()
all_feature_cols = X.columns.tolist()

feature_block_summary = pd.DataFrame(
    [
        {
            "feature_block" : "RDKit descriptors",
            "n_features" : len(rdkit_feature_cols),
        },
        {
            "feature_block" : "Morgan fingerprints",
            "n_features" : len(morgan_feature_cols),
        },
        {
            "feature_block" : "Combined X",
            "n_features" : len(all_feature_cols)
        }
    ]
)

feature_block_summary

,feature_block,n_features
0,RDKit descriptors,217
1,Morgan fingerprints,2048
2,Combined X,2265


**Feature Artifact Contract Summary**

The loaded feature artifacts match the expected Notebook 06 structure:

- RDKit descriptor block: 217 features
- Morgan fingerprint block: 2,048 features
- Combined feature matrix `X`: 2,265 features

Notebook 06 already confirmed that the assembled artifacts were row-aligned and that policy-forbidden raw columns did not enter `X`. Notebook 07 therefore treats the saved feature artifacts as frozen inputs and avoids repeating the full featurization contract.

The next QA step focuses on value validity: missing values, infinite values, and feature-generation issues that could affect downstream preprocessing or scaffold-aware modeling.

In [8]:
# compute missing value-counts and inf value counts for each feature block
# this was already done in Notebook 06 for missingness, just verifying here.
missing_summary = {
    "X": X.isna().sum().sum(),
    "rdkit_descriptors": rdkit_descriptors.isna().sum().sum(),
    "morgan_fingerprints": morgan_fingerprints.isna().sum().sum(),
}

infinite_summary = {
    "X": np.isinf(X.to_numpy()).sum(),
    "rdkit_descriptors": np.isinf(rdkit_descriptors.to_numpy()).sum(),
    "morgan_fingerprints": np.isinf(morgan_fingerprints.to_numpy()).sum(),
}

failed_descriptor_cols = (
    rdkit_descriptors
    .isna()
    .sum()
    .loc[lambda s: s > 0]
    .index
    .tolist()
)

has_missing_rdkit_descriptor = rdkit_descriptors.isna().any(axis=1)

feature_qa_flags = pd.DataFrame(
    {
        "has_missing_rdkit_descriptor": has_missing_rdkit_descriptor,
    },
    index=X.index,
)

value_quality_summary = pd.DataFrame(
    [
        {
            "artifact": artifact_name,
            "n_missing_values": missing_summary[artifact_name],
            "n_infinite_values": infinite_summary[artifact_name],
        }
        for artifact_name in missing_summary
    ]
)

display(value_quality_summary)
display(feature_qa_flags.sum().to_frame("n_flagged_molecules"))
print(f"Failed RDKit descriptor columns: {len(failed_descriptor_cols)}")

,artifact,n_missing_values,n_infinite_values
0,X,32,0
1,rdkit_descriptors,32,0
2,morgan_fingerprints,0,0


,n_flagged_molecules
has_missing_rdkit_descriptor,3


Failed RDKit descriptor columns: 12


**Value Validity QA Summary**

The feature-value audit confirmed the known Notebook 06 RDKit descriptor missingness:

- `X`: 32 missing values, 0 infinite values
- RDKit descriptors: 32 missing values, 0 infinite values
- Morgan fingerprints: 0 missing values, 0 infinite values

All missing feature values are localized to the RDKit descriptor block. The affected descriptor columns were retained in `failed_descriptor_cols`, and affected molecules were flagged in `feature_qa_flags`.

The missing RDKit values are treated as feature-generation edge cases requiring downstream preprocessing, not as grounds for molecule removal. Downstream model pipelines must handle RDKit missingness using fold-internal preprocessing during scaffold-aware cross-validation.

No infinite values were found in any feature block.

### **Section 4: Scaffold-Aware Training/Test Split Strategy**

This section creates Bemis--Murcko scaffold assignments from the saved molecular structure metadata and uses them to define a scaffold-aware training/test split.

Scaffold logic is used for evaluation design, not for basic feature QA. Notebook 07 creates the final scaffold-held-out test set. Notebook 08 and later modeling notebooks will perform model comparison and tuning using scaffold-aware cross-validation inside the training set.

The goal is to reduce chemical-series leakage: molecules sharing the same scaffold should not be split across training and final test partitions.

The final test set must not be used for model comparison, hyperparameter tuning, feature-set ablation, preprocessing-policy selection, or threshold/ranking-policy selection.

In [9]:
# create our var from our metadata used to create our scaffolds
# the source col will be canonical_smiles, which was our STRUCTURE_SOURCE we used in prev notebooks
STRUCTURE_COL = "canonical_smiles"

assert STRUCTURE_COL in metadata.columns, (
    f"Missing structure column in metadata: {STRUCTURE_COL}"
)

metadata[[STRUCTURE_COL]].head()

,canonical_smiles
0,CCOC(=O)c1ncn2c1CN(C)C(=O)c1cc(Cl)ccc1-2
1,CCOC(=O)c1ncn2c1CN(C)C(=O)c1cc(F)ccc1-2
2,CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21
3,CCOC(=O)c1ncn2c1CN(C)C(=O)c1cc(N=[N+]=[N-])ccc1-2
4,CN1Cc2c(C(=O)OC(C)(C)C)ncn2-c2ccc(N=[N+]=[N-])...


In [10]:
# generate a Bemis-Murcko scaffold SMILES from a molecule SMILES
# returns None if the molecule can't be parsed.

def smiles_to_bemis_murcko_scaffold(smiles : str) -> str | None:
    """
    Convert one molecule SMILES into a Bemis-Murcko scaffold SMILES.

    Returns
    ---------
    str | None
        Bemis-Murcko scaffold SMILES if parsing succeeds.
        None if this SMILES cannot be parsed.
    """
    # parse the SMILES string into an RDKit Mol object
    mol = Chem.MolFromSmiles(smiles)

    # return None if object failed
    if mol is None:
        return None
    
    # generate the Bemis-Murcko scaffold SMILES
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol = mol)

    return scaffold



In [11]:
# create scaffold assignments to X / metadata, apply the function.
scaffold_assignments = metadata[[STRUCTURE_COL]].copy()

scaffold_assignments["bemis_murcko_scaffold"] = (
    scaffold_assignments[STRUCTURE_COL]
    .apply(smiles_to_bemis_murcko_scaffold)
)

# count scaffold parsing failures
n_missing_scaffolds = scaffold_assignments["bemis_murcko_scaffold"].isna().sum()

assert n_missing_scaffolds == 0, (
    f"Unexpected missing scaffold assignments: {n_missing_scaffolds}"
)

scaffold_assignments.head()

,canonical_smiles,bemis_murcko_scaffold
0,CCOC(=O)c1ncn2c1CN(C)C(=O)c1cc(Cl)ccc1-2,O=C1NCc2cncn2-c2ccccc21
1,CCOC(=O)c1ncn2c1CN(C)C(=O)c1cc(F)ccc1-2,O=C1NCc2cncn2-c2ccccc21
2,CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21,O=C1CN=C(c2ccccc2)c2ccccc2N1
3,CCOC(=O)c1ncn2c1CN(C)C(=O)c1cc(N=[N+]=[N-])ccc1-2,O=C1NCc2cncn2-c2ccccc21
4,CN1Cc2c(C(=O)OC(C)(C)C)ncn2-c2ccc(N=[N+]=[N-])...,O=C1NCc2cncn2-c2ccccc21


The scaffold assignment table shows that multiple distinct molecules can share the same Bemis--Murcko scaffold. This confirms why splitting at the molecule-row level would risk chemical-series leakage. Downstream train/test assignment must occur at the scaffold-group level, not independently by molecule.

In [12]:
# summarize scaffold group sizes.
scaffold_group_sizes = (
    scaffold_assignments["bemis_murcko_scaffold"]
    .value_counts()
    .rename_axis("bemis_murcko_scaffold")
    .reset_index(name = "n_molecules")
)

n_unique_molecules = scaffold_group_sizes.shape[0]
largest_scaffold_size = scaffold_group_sizes["n_molecules"].max()
n_singleston_scaffolds = (scaffold_group_sizes["n_molecules"] == 1).sum()

print(f"Unique Bemis-Murcko scaffolds: {n_unique_molecules}")
print(f"Largest scaffold size: {largest_scaffold_size}")
print(f"Singleton scaffolds: {n_singleston_scaffolds}")

scaffold_group_sizes.head(10)

Unique Bemis-Murcko scaffolds: 565
Largest scaffold size: 59
Singleton scaffolds: 384


,bemis_murcko_scaffold,n_molecules
0,O=C1CN=C(c2ccccc2)c2ccccc2N1,59
1,c1ccc2c(c1)[nH]c1cnccc12,47
2,c1ccc2c(c1)-c1ncnn1Cc1cncn1-2,46
3,O=c1cc(-c2ccccc2)oc2ccccc12,42
4,O=C1NCc2cncn2-c2ccccc21,35
5,c1ccc(-c2ccccc2)cc1,32
6,O=C(NCc1ccccc1)C(=O)c1c[nH]c2ccccc12,29
7,c1ccccc1,27
8,c1ccc2c3cc[nH]cc-3nc2c1,26
9,c1ccc(-c2nnc3c4c(c(OCc5ccccn5)nn23)C2CCC4CC2)cc1,23


The dataset contains 565 unique Bemis--Murcko scaffolds across 1,546 molecules. Most scaffolds are singletons, while the largest scaffold contains 59 molecules. This distribution suggests that a scaffold-aware training/test split should be feasible without a single scaffold dominating the split.

In [13]:
# build a scaffold-level table for the split assignment
# each row represents one scaffold group, not a molecule here
primary_target_col = y_primary.columns[0]

scaffold_split_frame = (
    scaffold_assignments
    .join(y_primary)
    .groupby("bemis_murcko_scaffold")
    .agg(
        n_molecules = (primary_target_col, "size"),
        n_active = (primary_target_col, "sum"),
    )
    .reset_index()
)

# compute inactive count as a check
scaffold_split_frame["n_inactive"] = (
    scaffold_split_frame["n_molecules"] - scaffold_split_frame["n_active"]
)

# compute scaffold-level active rate
scaffold_split_frame["active_rate"] = (
    scaffold_split_frame["n_active"] / scaffold_split_frame["n_molecules"]
)

scaffold_split_frame.sort_values("n_molecules", ascending = False).head(10)

,bemis_murcko_scaffold,n_molecules,n_active,n_inactive,active_rate
163,O=C1CN=C(c2ccccc2)c2ccccc2N1,59,12,47,0.203390
498,c1ccc2c(c1)[nH]c1cnccc12,47,39,8,0.829787
486,c1ccc2c(c1)-c1ncnn1Cc1cncn1-2,46,44,2,0.956522
242,O=c1cc(-c2ccccc2)oc2ccccc12,42,29,13,0.690476
179,O=C1NCc2cncn2-c2ccccc21,35,29,6,0.828571
315,c1ccc(-c2ccccc2)cc1,32,1,31,0.031250
98,O=C(NCc1ccccc1)C(=O)c1c[nH]c2ccccc12,29,28,1,0.965517
531,c1ccccc1,27,0,27,0.000000
516,c1ccc2c3cc[nH]cc-3nc2c1,26,24,2,0.923077
341,c1ccc(-c2nnc3c4c(c(OCc5ccccn5)nn23)C2CCC4CC2)cc1,23,23,0,1.000000


**Split Size Policy**

BioPred uses a scaffold-aware training/test split rather than a fixed train/validation/test split. Model comparison and tuning will occur only inside the training set using scaffold-aware cross-validation, while the final scaffold-held-out test set is reserved for one-time final evaluation.

The initial target split is 80/20. This keeps most molecules available for training-set cross-validation while still reserving a meaningful scaffold-held-out test set. Because scaffolds are assigned as intact groups, the final split sizes may not match the target fraction exactly.

A larger 70/30 split would provide more final test molecules, but it would reduce the training pool used for scaffold-aware cross-validation. Given the modest dataset size, 80/20 is the preferred starting point.


In [14]:
# scaffold-aware split configuration
RANDOM_STATE = 42
TEST_SIZE = 0.2

target_test_molecules = int(round(len(X) * TEST_SIZE))
target_train_molecules = len(X) - target_test_molecules

print(f"Target train molecules: {target_train_molecules}")
print(f"Target test molecules: {target_test_molecules}")

Target train molecules: 1237
Target test molecules: 309


In [15]:
# accumulate whole scaffolds into test until target size is reached
# put rest into train

rng = np.random.default_rng(RANDOM_STATE)

# shuffle the scaffold-level rows
shuffled_scaffolds = (
    scaffold_split_frame
    .sample(frac = 1, random_state = RANDOM_STATE)
    .reset_index(drop = True)
)

test_scaffolds = []
test_molecule_count = 0

for _, row in shuffled_scaffolds.iterrows():
    # stop once adding more scaffolds exceeds the target too much
    # keep it simple at first then iterate
    if test_molecule_count >= target_test_molecules:
        break

    test_scaffolds.append(row["bemis_murcko_scaffold"])
    test_molecule_count += row["n_molecules"]

test_scaffolds = set(test_scaffolds)

scaffold_split_frame["split"] = np.where(
    scaffold_split_frame["bemis_murcko_scaffold"].isin(test_scaffolds),
    "test",
    "train"
)

scaffold_split_frame["split"].value_counts()

split
train    436
test     129
Name: count, dtype: int64

In [16]:
# map scaffold-level assignments back to molecule rows
scaffold_to_split = (
    scaffold_split_frame
    .set_index("bemis_murcko_scaffold")["split"]
)

split_assignments = scaffold_assignments.copy()

# map each molecule's scaffold to its assigned split.
split_assignments["split"] = (
    split_assignments["bemis_murcko_scaffold"]
    .map(scaffold_to_split)
)

# confirm every molecule received a split assignment
assert split_assignments["split"].notna().all(), (
    "Some molecules were not assigned a train/test split."
)

split_assignments["split"].value_counts()

split
train    1227
test      319
Name: count, dtype: int64

The scaffold-aware split produced 1,227 training molecules and 319 final test molecules. This is close to the target 80/20 split. Exact row counts are not expected because entire scaffold groups are assigned together and cannot be split across train and test.

In [17]:
# summarize primary-label balance by split

# combine split assignments with the primary target
split_label_frame = (
    split_assignments
    .join(y_primary)
)

# summarize row count, active count, and inactive count by split
split_label_summary = (
    split_label_frame
    .groupby("split")
    .agg(
        n_molecules = (primary_target_col, "size"),
        n_active = (primary_target_col, "sum"),
    )
    .reset_index()
)

# compute inactive count as a check
split_label_summary["n_inactive"] = (
    split_label_summary["n_molecules"] - split_label_summary["n_active"]
)

# compute scaffold-level active rate
split_label_summary["active_rate"] = (
    split_label_summary["n_active"] / split_label_summary["n_molecules"]
)

split_label_summary

,split,n_molecules,n_active,n_inactive,active_rate
0,test,319,252,67,0.789969
1,train,1227,948,279,0.772616


The scaffold-aware split preserved primary-label balance reasonably well. The training set has an active rate of approximately 77.3%, while the final test set has an active rate of approximately 79.0%. Both partitions contain active and inactive molecules, so the split is non-degenerate for binary model evaluation and ranking analysis.

In [18]:
# validate that no scaffolds appear in both the train and test sets
# this is the core leakage-control invariant for the final held-out split

# collect the set of scaffolds assigned to train
train_scaffolds = set(
    split_assignments
    .loc[split_assignments["split"] == "train", "bemis_murcko_scaffold"]
)

# collect the set of scaffolds assigned to test
test_scaffolds = set(
    split_assignments
    .loc[split_assignments["split"] == "test", "bemis_murcko_scaffold"]
)

# compute overlap between train and test scaffolds
scaffold_overlap = train_scaffolds.intersection(test_scaffolds)

assert len(scaffold_overlap) == 0, (
    f"Scaffold leakage detected: {len(scaffold_overlap)} overlapping scaffolds."
)

print(f"Train scaffolds: {len(train_scaffolds)}")
print(f"Test scaffolds: {len(test_scaffolds)}")
print(f"Overlapping scaffolds: {len(scaffold_overlap)}")

Train scaffolds: 436
Test scaffolds: 129
Overlapping scaffolds: 0


The scaffold-overlap check passed. The training and final test partitions contain disjoint Bemis-Murcko scaffold sets: 436 scaffolds in training, 129 scaffolds in test, and 0 overlapping scaffolds. This satisfies the core leakage-control requirement for the final scaffold-held-out test set.

In [19]:
# join split assignments with row-level feature QA flags.
split_qa_flag_frame = (
    split_assignments[["split"]]
    .join(feature_qa_flags)
)

# summarize flagged molecule counts by split
split_qa_flag_summary = (
    split_qa_flag_frame
    .groupby("split")
    .agg(
        n_molecules = ("split", "size"),
        n_missing_rdkit_descriptor = ("has_missing_rdkit_descriptor", "sum")
    )
)

# compute rate of missing RDKit flag by split
split_qa_flag_summary["missing_rdkit_descriptor_rate"] = (
    split_qa_flag_summary["n_missing_rdkit_descriptor"] / split_qa_flag_summary["n_molecules"]
)

split_qa_flag_summary

,n_molecules,n_missing_rdkit_descriptor,missing_rdkit_descriptor_rate
split,,,
test,319,3,0.009404
train,1227,0,0.000000


In [20]:
# inspect the subject molecules with missing RDKit descriptor values after split assignment.
# Goal is to determine whether the three flagged molecules belong to one scaffold group or multiple.
flagged_molecule_review = (
    split_assignments
    .join(feature_qa_flags)
    .join(metadata[["molregno", "molecule_chembl_id"]])
)

flagged_molecule_review = flagged_molecule_review.loc[
    flagged_molecule_review["has_missing_rdkit_descriptor"]
]

flagged_molecule_review[
    [
        "split",
        "bemis_murcko_scaffold",
        "has_missing_rdkit_descriptor",
        "molregno",
        "molecule_chembl_id",
        STRUCTURE_COL,
    ]
]




,split,bemis_murcko_scaffold,has_missing_rdkit_descriptor,molregno,molecule_chembl_id,canonical_smiles
117,test,,True,59510,CHEMBL41065,NCCC[Se](=O)O
124,test,C1CCNCC1,True,60410,CHEMBL290961,O=[Se](O)C1CCNCC1
660,test,N=C(c1ccccc1)c1ccccc1,True,268340,CHEMBL160058,O=C([O-])CCC/N=C(\c1ccc(Cl)cc1)c1cc(F)ccc1O.[Na+]


**Split QA Finding: Feature-Flag Concentration**

The initial scaffold-aware 80/20 split produced a valid scaffold-held-out test set with no scaffold overlap between training and test. Primary-label balance was also acceptable: the training active rate was approximately 77.3%, and the test active rate was approximately 79.0%.

However, all 3 molecules with missing RDKit descriptor values landed in the final test set. These molecules are retained because the missing values are known RDKit descriptor edge cases, not invalid molecule records. Still, their concentration in the test set is worth addressing because the training set would contain no examples of this feature-missingness pattern.

This does not mean the split is invalid. It means the split should be improved if possible using pre-modeling split-quality criteria. We may search across random scaffold split seeds and choose a split that preserves:

- zero scaffold overlap
- approximately 80/20 molecule balance
- non-degenerate active/inactive labels in both partitions
- reasonable primary-label balance
- less pathological distribution of feature-QA flags

The split seed must not be selected based on model performance. Seed selection is allowed here only because it uses pre-modeling data-quality and split-quality diagnostics.

In [21]:
# make a function this time for making the scaffold split
# we have already done this manually, but now we need something reusable for seed swapping.

def make_scaffold_split(scaffold_table, target_test_n, seed):
    """
    Assign scaffold groups to train/test using a randomized scaffold-level order.

    Returns
    -------
    pd.Series
        Index: Bemis--Murcko scaffold SMILES
        Values: "train" or "test"
    """

    shuffled_table = (
        scaffold_table
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    selected_test_scaffolds = []
    selected_test_n = 0

    for _, scaffold_row in shuffled_table.iterrows():
        if selected_test_n >= target_test_n:
            break

        selected_test_scaffolds.append(scaffold_row["bemis_murcko_scaffold"])
        selected_test_n += scaffold_row["n_molecules"]

    selected_test_scaffolds = set(selected_test_scaffolds)

    split_by_scaffold = pd.Series(
        data=np.where(
            scaffold_table["bemis_murcko_scaffold"].isin(selected_test_scaffolds),
            "test",
            "train",
        ),
        index=scaffold_table["bemis_murcko_scaffold"],
        name="split",
    )

    return split_by_scaffold



In [22]:
candidate_split_rows = []

for seed in range(50):
    split_by_scaffold = make_scaffold_split(
        scaffold_table=scaffold_split_frame.drop(columns=["split"], errors="ignore"),
        target_test_n=target_test_molecules,
        seed=seed,
    )

    candidate_split_assignments = scaffold_assignments.copy()

    candidate_split_assignments["split"] = (
        candidate_split_assignments["bemis_murcko_scaffold"]
        .map(split_by_scaffold)
    )

    assert candidate_split_assignments["split"].notna().all(), (
        f"Missing split assignment for seed {seed}."
    )

    candidate_eval_frame = (
        candidate_split_assignments[["split", "bemis_murcko_scaffold"]]
        .join(y_primary)
        .join(feature_qa_flags)
    )

    split_summary = (
        candidate_eval_frame
        .groupby("split")
        .agg(
            n_molecules=(primary_target_col, "size"),
            n_active=(primary_target_col, "sum"),
            n_missing_rdkit_descriptor=("has_missing_rdkit_descriptor", "sum"),
        )
    )

    split_summary["active_rate"] = (
        split_summary["n_active"] / split_summary["n_molecules"]
    )

    train_scaffolds = set(
        candidate_eval_frame.loc[
            candidate_eval_frame["split"] == "train",
            "bemis_murcko_scaffold",
        ]
    )

    test_scaffolds = set(
        candidate_eval_frame.loc[
            candidate_eval_frame["split"] == "test",
            "bemis_murcko_scaffold",
        ]
    )

    candidate_split_rows.append(
        {
            "seed": seed,
            "train_n": split_summary.loc["train", "n_molecules"],
            "test_n": split_summary.loc["test", "n_molecules"],
            "train_active_rate": split_summary.loc["train", "active_rate"],
            "test_active_rate": split_summary.loc["test", "active_rate"],
            "active_rate_abs_diff": abs(
                split_summary.loc["train", "active_rate"]
                - split_summary.loc["test", "active_rate"]
            ),
            "train_missing_rdkit_n": split_summary.loc[
                "train", "n_missing_rdkit_descriptor"
            ],
            "test_missing_rdkit_n": split_summary.loc[
                "test", "n_missing_rdkit_descriptor"
            ],
            "scaffold_overlap_n": len(train_scaffolds.intersection(test_scaffolds)),
        }
    )

candidate_split_summary = pd.DataFrame(candidate_split_rows)

candidate_split_summary.sort_values(
    [
        "scaffold_overlap_n",
        "active_rate_abs_diff",
        "test_missing_rdkit_n",
    ]
).head(10)

,seed,train_n,test_n,train_active_rate,test_active_rate,active_rate_abs_diff,train_missing_rdkit_n,test_missing_rdkit_n,scaffold_overlap_n
33,33,1236,310,0.775890,0.777419,0.001529,2,1,0
12,12,1191,355,0.775819,0.777465,0.001646,2,1,0
32,32,1231,315,0.774980,0.780952,0.005973,3,0,0
10,10,1228,318,0.772801,0.789308,0.016507,3,0,0
45,45,1237,309,0.772838,0.789644,0.016807,2,1,0
42,42,1227,319,0.772616,0.789969,0.017353,0,3,0
28,28,1233,313,0.780211,0.760383,0.019827,3,0,0
11,11,1236,310,0.780744,0.758065,0.022680,3,0,0
23,23,1215,331,0.781070,0.758308,0.022762,3,0,0
30,30,1226,320,0.781403,0.756250,0.025153,2,1,0


**Bounded Seed Check Results**

The bounded seed check compared candidate scaffold-aware train/test splits using only pre-modeling split-quality diagnostics. No model performance was used.

The initial seed `42` produced a valid scaffold-disjoint split, but all 3 molecules with missing RDKit descriptor values landed in the final test set. This was not a fatal issue because the affected count was small, but it was worth checking whether a cleaner split was available without harming the main validation criteria.

Seed `33` was selected because it improved the split without introducing a meaningful tradeoff:

- train/test molecule count: 1,236 / 310
- active-rate balance: approximately 77.6% in train and 77.7% in test
- active-rate absolute difference: approximately 0.0015
- RDKit-missing descriptor flags: 2 in train and 1 in test
- scaffold overlap: 0

Seed `33` is preferred over the initial seed `42` because it keeps the split close to the intended 80/20 design, improves primary-label balance, avoids concentrating all RDKit-missing molecules in the test set, and preserves scaffold disjointness.

This is a bounded split-quality adjustment, not split optimization. The selected seed is based only on dataset structure, labels, and QA flags available before modeling. The final test set remains untouched for model comparison, tuning, preprocessing-policy selection, and threshold/ranking-policy selection.

**Next Actions**

The selected seed will now be applied and locked as the official Notebook 07 scaffold-aware training/test split. After locking the split, the notebook will rerun the final split QA summaries using seed `33`, including:

- train/test molecule counts
- primary-label balance
- scaffold overlap check
- feature-QA flag distribution
- scaffold concentration summary

Once the locked split passes these checks, the notebook will move to downstream preprocessing policy and artifact export. The remaining work is mostly mechanical: create train/test masks, partition raw features and labels, save split-aware modeling artifacts, and write the Notebook 08 handoff policy.

In [23]:
# now we choose our new seed (33) and overwrite/lock it as the new seed in our scaffold splits going forward
SELECTED_SPLIT_SEED = 33

split_by_scaffold = make_scaffold_split(
    scaffold_table=scaffold_split_frame.drop(columns = ["split"], errors = "ignore"),
    target_test_n=target_test_molecules,
    seed = SELECTED_SPLIT_SEED,
)

# apply the scaffold-level split back to molecule-level rows
split_assignments = scaffold_assignments.copy()

split_assignments["split"] = (
    split_assignments["bemis_murcko_scaffold"]
    .map(split_by_scaffold)
)

# confirm every molecule received a split assignment
assert split_assignments["split"].notna().all(), (
    "Some molecules were not assigned a train/test split."
)

# summarize locked split sizes
split_assignments["split"].value_counts()

split
train    1236
test      310
Name: count, dtype: int64

**Section 4 Summary**

Section 4 created the scaffold-aware training/test split used for downstream modeling.

Bemis--Murcko scaffolds were generated from the saved `canonical_smiles` metadata. The dataset contained 565 unique scaffolds across 1,546 molecules, with 384 singleton scaffolds and a largest scaffold size of 59 molecules. Because multiple molecules can share the same scaffold, splitting by molecule row alone would risk chemical-series leakage.

The initial 80/20 scaffold split using seed `42` produced a valid scaffold-disjoint split, but all 3 molecules with missing RDKit descriptor values landed in the final test set. A bounded seed check was therefore performed using only pre-modeling split-quality diagnostics: molecule counts, active-rate balance, scaffold overlap, and feature-QA flag distribution. No model performance was used.

Seed `33` was selected and locked as the final split seed. The locked split contains 1,236 training molecules and 310 final test molecules, with zero scaffold overlap. The split also improves label balance and distributes the 3 RDKit-missing descriptor cases as 2 in training and 1 in test.

This locked split defines the evaluation boundary for the rest of BioPred. Notebook 08 will perform model comparison and tuning only inside the training set, while the scaffold-held-out test set is reserved for one-time final evaluation.

### **Section 5: Split Quality Validation**

This section validates the locked scaffold-aware training/test split created in Section 4.

The purpose is not to optimize the split further. The selected seed has already been locked using pre-modeling split-quality diagnostics. This section records the final validation checks that will be saved into the Notebook 07 split QA report and used to support downstream modeling claims.

The locked split is evaluated on:

- train/test molecule counts
- primary-label balance
- scaffold overlap between train and test
- feature-QA flag distribution
- scaffold concentration within each partition
- sensitivity-label balance

These checks establish that the final test set is scaffold-held-out, non-degenerate, reasonably balanced, and suitable for one-time final model evaluation. Model comparison and tuning will still occur only inside the training set in Notebook 08.

In [24]:
# Validate primary-label balance for the locked scaffold-aware split.

locked_split_label_frame = (
    split_assignments
    .join(y_primary)
)

locked_split_label_summary = (
    locked_split_label_frame
    .groupby("split")
    .agg(
        n_molecules=(primary_target_col, "size"),
        n_active=(primary_target_col, "sum"),
    )
)

locked_split_label_summary["n_inactive"] = (
    locked_split_label_summary["n_molecules"]
    - locked_split_label_summary["n_active"]
)

locked_split_label_summary["active_rate"] = (
    locked_split_label_summary["n_active"]
    / locked_split_label_summary["n_molecules"]
)

locked_split_label_summary

,n_molecules,n_active,n_inactive,active_rate
split,,,,
test,310,241,69,0.777419
train,1236,959,277,0.775890


In [27]:
# confirm scaffold disjointness for the locked split.
# in this validation check no scaffold should appear in both train and test.

# collect the set of scaffolds assigned to train.
locked_train_scaffolds = set(
    split_assignments.loc[
        split_assignments["split"] == "train", "bemis_murcko_scaffold"
    ]
)

# collect the set of scaffolds assigned to test.
locked_test_scaffolds = set(
    split_assignments.loc[
        split_assignments["split"] == "test", "bemis_murcko_scaffold"
    ]
)

# compute scaffold overlap
locked_scaffold_overlap = locked_train_scaffolds.intersection(locked_test_scaffolds)

# assertion statement that there is no overlap
assert len(locked_scaffold_overlap) == 0, (
    f"Scaffold leakage detected: {len(locked_scaffold_overlap)} overlapping scaffolds."
)

print(f"Train scaffolds: {len(locked_train_scaffolds)}")
print(f"Test scaffolds: {len(locked_test_scaffolds)}")
print(f"Overlapping scaffolds: {len(locked_scaffold_overlap)}")



Train scaffolds: 440
Test scaffolds: 125
Overlapping scaffolds: 0


The locked split contains 440 training scaffolds and 125 test scaffolds, with 0 overlapping scaffolds. This confirms that the final test set is scaffold-held-out and that no Bemis--Murcko scaffold appears in both training and test.

In [30]:
# summarize row-level feature QA flags by locked split.
# want to confirm known RDKit descriptor missingness is visible and documented after split locking.

# join locked split labels with row-level feature QA flags.
locked_split_level_qa_flag_frame = (
    split_assignments[["split"]]
    .join(feature_qa_flags)
)

# summarize molecule counts and missing-RDKit counts by split.
locked_split_qa_flag_summary = (
    locked_split_level_qa_flag_frame
    .groupby("split")
    .agg(
        n_molecules = ("split", "size"),
        n_missing_rdkit_descriptor = ("has_missing_rdkit_descriptor", "sum"),
    )
)

# compute missing-RDKit flag rate by split
locked_split_qa_flag_summary["missing_rdkit_descriptor_rate"] = (
    locked_split_qa_flag_summary["n_missing_rdkit_descriptor"] / 
    locked_split_qa_flag_summary["n_molecules"]
)

locked_split_qa_flag_summary

,n_molecules,n_missing_rdkit_descriptor,missing_rdkit_descriptor_rate
split,,,
test,310,1,0.003226
train,1236,2,0.001618


The locked split distributes the known RDKit descriptor-missing cases across both partitions: 2 molecules in training and 1 molecule in test. The affected rates are low in both partitions: approximately 0.16% in training and 0.32% in test.

These molecules are retained and flagged rather than removed. Downstream preprocessing must handle RDKit descriptor missingness inside each scaffold-aware cross-validation fold and inside the final training pipeline before final test evaluation.

In [31]:
# summarize scaffold concentraion within each locked split
# check whether train or test is dominated by unusally large scaffold groups.

# count molecules per scaffold within each split.
locked_split_scaffold_sizes = (
    split_assignments
    .groupby(["split", "bemis_murcko_scaffold"])
    .size()
    .reset_index(name = "n_molecules")
)

# summarize scaffold concentration by split.
locked_scaffold_concentration_summary = (
    locked_split_scaffold_sizes
    .groupby("split")
    .agg(
        n_scaffolds = ("bemis_murcko_scaffold", "nunique"),
        largest_scaffold_size = ("n_molecules", "max"),
        median_scaffold_size = ("n_molecules", "median"),
    )
)

#get molecule counts by split
locked_split_sizes = split_assignments["split"].value_counts()

# compute largest scaffold fraction within each split
locked_scaffold_concentration_summary["largest_scaffold_fraction"] = (
    locked_scaffold_concentration_summary["largest_scaffold_size"] / 
    locked_scaffold_concentration_summary.index.map(locked_split_sizes)
)

locked_scaffold_concentration_summary

,n_scaffolds,largest_scaffold_size,median_scaffold_size,largest_scaffold_fraction
split,,,,
test,125,27,1.0,0.087097
train,440,59,1.0,0.047735


The locked split does not appear to be dominated by a single scaffold group. The largest training scaffold contains 59 molecules, representing approximately 4.8% of the training set. The largest test scaffold contains 27 molecules, representing approximately 8.7% of the test set.

The median scaffold size is 1 in both partitions, indicating that most scaffold groups are singletons. The scaffold concentration profile is acceptable for downstream modeling handoff.

In [34]:
# summarize sensitivity-label balance for the locked split.
# confirm secondary label definitions remain usable in both partitions.

# join locked split labels with sensitivity labels.
locked_sensitivity_label_frame = (
    split_assignments[["split"]]
    .join(y_sensitivity)
)

# compute mean label rate by split
locked_sensitivity_rate_summary = (
    locked_sensitivity_label_frame
    .groupby("split")
    .mean()
)

# compute active counts by split
locked_sensitivity_count_summary = (
    locked_sensitivity_label_frame
    .groupby("split")
    .sum()
)

display(locked_sensitivity_count_summary)
display(locked_sensitivity_rate_summary)

,active_median_px_5_5,active_median_px_6_5
split,,
test,254,214
train,1044,831


,active_median_px_5_5,active_median_px_6_5
split,,
test,0.819355,0.690323
train,0.844660,0.672330


The sensitivity-label balance check passed. Both secondary label definitions remain represented in training and test.

For `active_median_px_5_5`, the active rate is approximately 84.5% in training and 81.9% in test. For `active_median_px_6_5`, the active rate is approximately 67.2% in training and 69.0% in test. These differences are acceptable for downstream sensitivity analysis.

This confirms that the locked scaffold-aware split preserves usable primary and secondary label distributions while maintaining scaffold disjointness.

**Section 5 Summary**

Section 5 validated the locked scaffold-aware training/test split created in Section 4.

The final split contains 1,236 training molecules and 310 final test molecules. Primary-label balance is strong, with active rates of approximately 77.6% in training and 77.7% in test. The scaffold-overlap check confirmed 440 training scaffolds, 125 test scaffolds, and 0 overlapping scaffolds.

Known RDKit descriptor-missing cases are retained and flagged. The locked split assigns 2 of these molecules to training and 1 to test, avoiding concentration of all feature-generation edge cases in the final test set.

Scaffold concentration is acceptable. The largest training scaffold represents approximately 4.8% of the training set, and the largest test scaffold represents approximately 8.7% of the test set. The median scaffold size is 1 in both partitions.

Sensitivity-label balance is also acceptable. Both secondary label definitions remain represented in training and test, supporting downstream robustness checks.

The locked split is suitable for modeling handoff. Notebook 08 should use only the training set for model comparison, hyperparameter tuning, preprocessing-policy fitting, and cross-validation. The scaffold-held-out test set remains reserved for one-time final model evaluation.

### **Section 6: Downstream Preprocessing Policy**

This section records how downstream modeling notebooks should preprocess the saved feature artifacts.

Notebook 07 does not fit imputers, scalers, variance filters, or other preprocessing objects. Those steps must be fit inside each scaffold-aware cross-validation training fold in Notebook 08. This prevents preprocessing leakage from validation folds or the final test set.

The preprocessing policy separates the two main feature blocks:

* RDKit descriptors are continuous molecular descriptors and may require missing-value handling, scaling, or model-specific preprocessing.
* Morgan fingerprints are binary molecular fingerprints and should usually pass through unchanged unless a downstream model pipeline applies fold-internal feature filtering.

The goal is to save a clear policy for Notebook 08 without making modeling decisions in this notebook.


In [36]:
# define downstream preprocessing policy.
# Notebook 07 records the policy but does not fit preprocessing objects.

preprocessing_policy = {
    "notebook" : "07_feature_qa_scaffold_split_modeling_handoff",
    "policy_scope" : "Downstream modeling policy only; no preprocessing artifacts are fit in Notebook 07.",
    "rdkit_descriptors" : {
        "n_features" : len(rdkit_feature_cols),
        "n_missing_values" : int(missing_summary["rdkit_descriptors"]),
        "n_molecules_with_missing_values" : int(
            feature_qa_flags["has_missing_rdkit_descriptor"].sum()
        ),
        "missing_descriptor_columns" : failed_descriptor_cols,
        "policy" : "Impute missing RDKit descriptor values inside each scaffold-aware CV training fold."
    },
    "morgan_fingerprints" : {
        "n_features" : len(morgan_feature_cols),
        "encoding" : "binary",
        "policy" : "Pass through by default; any filtering must be fit inside each scaffold-aware CV training fold."
    },
    "leakage_controls" : {
        "fit_preprocessing_in_notebook_07" : False,
        "fit_preprocessing_on_full_training_before_cv" : False,
        "fit_preprocessing_on_final_test" : False,
        "model_selection_scope" : "Training set only.",
        "final_test_use" : "One-time final evaluation only.",
    },
}

preprocessing_policy

{'notebook': '07_feature_qa_scaffold_split_modeling_handoff',
 'policy_scope': 'Downstream modeling policy only; no preprocessing artifacts are fit in Notebook 07.',
 'rdkit_descriptors': {'n_features': 217,
  'n_missing_values': 32,
  'n_molecules_with_missing_values': 3,
  'missing_descriptor_columns': ['rdkit_MaxPartialCharge',
   'rdkit_MinPartialCharge',
   'rdkit_MaxAbsPartialCharge',
   'rdkit_MinAbsPartialCharge',
   'rdkit_BCUT2D_MWHI',
   'rdkit_BCUT2D_MWLOW',
   'rdkit_BCUT2D_CHGHI',
   'rdkit_BCUT2D_CHGLO',
   'rdkit_BCUT2D_LOGPHI',
   'rdkit_BCUT2D_LOGPLOW',
   'rdkit_BCUT2D_MRHI',
   'rdkit_BCUT2D_MRLOW'],
  'policy': 'Impute missing RDKit descriptor values inside each scaffold-aware CV training fold.'},
 'morgan_fingerprints': {'n_features': 2048,
  'encoding': 'binary',
  'policy': 'Pass through by default; any filtering must be fit inside each scaffold-aware CV training fold.'},
 'leakage_controls': {'fit_preprocessing_in_notebook_07': False,
  'fit_preprocessing_on_fu

**Section 6 Summary**

Section 6 defines the downstream preprocessing policy without fitting any preprocessing artifacts in Notebook 07.

RDKit descriptors contain 32 missing values across 3 molecules and 12 descriptor columns. These values will be handled in Notebook 08 using fold-internal imputation during scaffold-aware cross-validation.

Morgan fingerprints are binary features and will pass through by default. Any optional filtering must also be fit inside each cross-validation training fold.

The final test set must not be used for preprocessing, model selection, or tuning. It remains reserved for one-time final evaluation after the modeling workflow is finalized.

### **Section 7: Save Split-Aware Modeling Artifacts**

This section saves the locked training/test artifacts for downstream modeling.

The saved artifacts remain raw and unprocessed. Notebook 07 does not impute missing values, scale descriptors, filter features, train models, or fit any preprocessing objects. Those steps belong inside the modeling pipelines used in Notebook 08.

The goal is to create a clean handoff package containing the split-aware features, labels, metadata, scaffold assignments, QA flags, feature schema, and preprocessing policy needed for scaffold-aware cross-validation and baseline modeling.


In [37]:
# create boolean masks for the locked train/test splits.
# these masks will be used to slice all split-aware modeling artifacts.

# create the training mask.
train_mask = split_assignments["split"].eq("train")

# create the test mask.
test_mask = split_assignments["split"].eq("test")

#confirm masks are mutually exclusive
assert not (train_mask & test_mask).any(), (
    "Train and test masks overlap."
)

# confirm every molecule is assigned to exactly one split.
assert (train_mask | test_mask).all(), (
    "Some molecules are not assigned to train or test."
)

print(f"Training molecules: {train_mask.sum()}")
print(f"Test molecules: {test_mask.sum()}")
print(f"Total molecules: {(train_mask | test_mask).sum()}")

Training molecules: 1236
Test molecules: 310
Total molecules: 1546


In [39]:
# slice raw artifacts into locked train/test partitions.
# these artifiacts remain unprocessed.

# split combined feature matrix.
X_train_raw = X.loc[train_mask].copy()
X_test_raw = X.loc[test_mask].copy()

# split primary target
y_train = y_primary.loc[train_mask].copy()
y_test = y_primary.loc[test_mask].copy()

# split sensitivity labels
y_sensitivity_train = y_sensitivity.loc[train_mask].copy()
y_sensitivity_test = y_sensitivity.loc[test_mask].copy()

# split metadata
metadata_train = metadata.loc[train_mask].copy()
metadata_test = metadata.loc[test_mask].copy()

# split feature qa flags
feature_qa_flags_train = feature_qa_flags.loc[train_mask].copy()
feature_qa_flags_test = feature_qa_flags.loc[test_mask].copy()

# split scaffold/split assignment table
split_assignments_train = split_assignments.loc[train_mask].copy()
split_assignments_test = split_assignments.loc[test_mask].copy()


print(f"X_train_raw: {X_train_raw.shape}")
print(f"X_test_raw: {X_test_raw.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

X_train_raw: (1236, 2265)
X_test_raw: (310, 2265)
y_train: (1236, 1)
y_test: (310, 1)


In [41]:
# validate row alignment after train/test slicing.
# every train artifact should share the same index; every test artifact should share the same index.

# collect train artifacts
train_artifacts = {
    "X_train_raw" : X_train_raw,
    "y_train" : y_train,
    "y_sensitivity_train" : y_sensitivity_train,
    "metadata_train" : metadata_train,
    "feature_qa_flags_train" : feature_qa_flags_train,
    "split_assignments_train" : split_assignments_train,
}

# collect test artifacts
test_artifacts = {
    "X_test_raw" : X_test_raw,
    "y_test" : y_test,
    "y_sensitivity_test" : y_sensitivity_test,
    "metadata_test" : metadata_test,
    "feature_qa_flags_test" : feature_qa_flags_test,
    "split_assignments_test" : split_assignments_test
}

# confirm train artifact alignment against X_train_raw.
for artifact_name, artifact in train_artifacts.items():
    assert artifact.index.equals(X_train_raw.index), (
        f"Train artifact index mismatch: {artifact_name}"
    )

# confirm test artifact alignment against X_test_raw.
for artifact_name, artifact in test_artifacts.items():
    assert artifact.index.equals(X_test_raw.index), (
        f"Test artifact index mismatch: {artifact_name}"
    )

print("Train/test artifact alignment checks passed.")


Train/test artifact alignment checks passed.


In [43]:
# Define feature schema artifact for downstream notebooks.
# Reuses feature-block columns established during artifact contract checks.

feature_schema = {
    "n_total_features": len(all_feature_cols),
    "feature_order": "RDKit descriptor columns followed by Morgan fingerprint columns.",
    "rdkit_descriptors": {
        "n_features": len(rdkit_feature_cols),
        "columns": rdkit_feature_cols,
    },
    "morgan_fingerprints": {
        "n_features": len(morgan_feature_cols),
        "columns": morgan_feature_cols,
    },
    "primary_target": primary_target_col,
    "sensitivity_targets": y_sensitivity.columns.tolist(),
}

print(f"Total features: {feature_schema['n_total_features']}")
print(f"RDKit descriptors: {feature_schema['rdkit_descriptors']['n_features']}")
print(f"Morgan fingerprints: {feature_schema['morgan_fingerprints']['n_features']}")

Total features: 2265
RDKit descriptors: 217
Morgan fingerprints: 2048


In [44]:
# define output paths for our modeling artifacts.

# make sure modeling output directory exists.  quick verification check.
paths.MODELING_DIR.mkdir(parents = True, exist_ok= True)

# define parquet artifact paths
modeling_artifact_paths = {
    "X_train_raw" : paths.MODELING_DIR / "X_train_raw.parquet",
    "X_test_raw" : paths.MODELING_DIR / "X_test_raw.parquet",
    "y_train" : paths.MODELING_DIR / "y_train.parquet",
    "y_test" : paths.MODELING_DIR / "y_test.parquet",
    "y_sensitivity_train" : paths.MODELING_DIR / "y_sensitivity_train.parquet",
    "y_sensitivity_test" : paths.MODELING_DIR / "y_sensitivity_test.parquet",
    "metadata_train" : paths.MODELING_DIR / "metadata_train.parquet",
    "metadata_test" : paths.MODELING_DIR / "metadata_test.parquet",
    "feature_qa_flags_train" : paths.MODELING_DIR / "feature_qa_flags_train.parquet",
    "feature_qa_flags_test" : paths.MODELING_DIR / "feature_qa_flags_test.parquet",
    "split_assignments" : paths.MODELING_DIR / "split_assignments.parquet",
    "split_assignments_train" : paths.MODELING_DIR / "split_assignments_train",
    "split_assignments_test" : paths.MODELING_DIR / "split_assignments_test.parquet",
}

# define JSON artifact path
modeling_policy_paths = {
    "feature_schema" : paths.MODELING_DIR / "feature_schema.json",
    "preprocessing_policy" : paths.MODELING_DIR / "preprocessing_policy.json",
}

# display paths for review
for artifact_name, artifact_path in {**modeling_artifact_paths, **modeling_policy_paths}.items():
    print(f"{artifact_name}: {artifact_path}")

X_train_raw: /home/ryanm/projects/BioPred/data/processed/modeling/X_train_raw.parquet
X_test_raw: /home/ryanm/projects/BioPred/data/processed/modeling/X_test_raw.parquet
y_train: /home/ryanm/projects/BioPred/data/processed/modeling/y_train.parquet
y_test: /home/ryanm/projects/BioPred/data/processed/modeling/y_test.parquet
y_sensitivity_train: /home/ryanm/projects/BioPred/data/processed/modeling/y_sensitivity_train.parquet
y_sensitivity_test: /home/ryanm/projects/BioPred/data/processed/modeling/y_sensitivity_test.parquet
metadata_train: /home/ryanm/projects/BioPred/data/processed/modeling/metadata_train.parquet
metadata_test: /home/ryanm/projects/BioPred/data/processed/modeling/metadata_test.parquet
feature_qa_flags_train: /home/ryanm/projects/BioPred/data/processed/modeling/feature_qa_flags_train.parquet
feature_qa_flags_test: /home/ryanm/projects/BioPred/data/processed/modeling/feature_qa_flags_test.parquet
split_assignments: /home/ryanm/projects/BioPred/data/processed/modeling/split_

In [46]:
# save our parquet artifacts
parquet_artifacts = {
    "X_train_raw" : X_train_raw,
    "X_test_raw" : X_test_raw,
    "y_train" : y_train,
    "y_test" : y_test,
    "y_sensitivity_train" : y_sensitivity_train,
    "y_sensitivity_test" : y_sensitivity_test,
    "metadata_train" : metadata_train,
    "metadata_test" : metadata_test,
    "feature_qa_flags_train" : feature_qa_flags_train,
    "feature_qa_flags_test" : feature_qa_flags_test,
    "split_assignments" : split_assignments,
    "split_assignments_train" : split_assignments_train,
    "split_assignments_test" : split_assignments_test
}

for artifact_name, artifact_df in parquet_artifacts.items():
    artifact_df.to_parquet(modeling_artifact_paths[artifact_name])

# save our JSON policy/schema artifacts
json_artifacts = {
    "feature_schema" : feature_schema,
    "preprocessing_policy" : preprocessing_policy,
}

for artifact_name, artifact_object in json_artifacts.items():
    with open(modeling_policy_paths[artifact_name], "w") as f:
        json.dump(artifact_object, f, indent = 2)

print("Split-aware modeling artifacts saved.")

Split-aware modeling artifacts saved.


In [47]:
# check and verify the artifacts are there before we conclude so we can reuse them
saved_artifact_paths = {
    **modeling_artifact_paths,
    **modeling_policy_paths,
}

missing_saved_artifacts = [
    artifact_name
    for artifact_name, artifact_path in saved_artifact_paths.items()
    if not artifact_path.exists()
]

assert not missing_saved_artifacts, (
    f"Missing saved artifacts: {missing_saved_artifacts}"
)

print(f"Saved artifacts checked: {len(saved_artifact_paths)}")
print("All expected modeling handoff artifacts exist.")

Saved artifacts checked: 15
All expected modeling handoff artifacts exist.


**Section 7 Summary**

Section 7 saved the locked split-aware modeling artifacts for Notebook 08.

The saved package includes raw training/test feature matrices, primary labels, sensitivity labels, metadata, feature-QA flags, scaffold split assignments, the feature schema, and the downstream preprocessing policy.

No preprocessing artifacts were fit in this notebook. The saved feature matrices remain raw, and downstream modeling notebooks must apply any imputation, scaling, or feature filtering inside scaffold-aware cross-validation pipelines.

These artifacts define the official Notebook 08 modeling inputs.

### **Section 8: Notebook 08 Handoff**

Notebook 07 saved the raw split-aware modeling artifacts needed for Notebook 08. The saved training artifacts should be used for baseline modeling, class-balance review, candidate model comparison, preprocessing-pipeline fitting, and scaffold-aware cross-validation.

The final test artifacts should not be used in Notebook 08 for model selection, hyperparameter tuning, preprocessing decisions, feature-pruning decisions, threshold selection, or ranking-policy selection.

Notebook 08 should load the raw training artifacts, apply preprocessing inside each scaffold-aware cross-validation training fold, and report baseline modeling results using the training set only. The scaffold-held-out test set remains reserved for one-time final evaluation in a later notebook.

Notebook 07 is now complete. It converted the frozen Notebook 06 feature artifacts into leakage-aware, split-aware modeling inputs while preserving the final test set for future evaluation.